# go-evo — Phase 2: evolving 10-layer MLP Go players on 19×19

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrimozRavbar/go-evo/blob/main/notebooks/phase2_colab.ipynb)

* 19×19 board, komi 7.5, move cap 3·N² = 1083. Children = Gaussian mutation of one parent, **no crossover**. Parents survive (elitism).
* **Phase 2a** (`sel_s*`, `ctl_s*`): 10 genomes, 5 round-robin games each, genomes with a winning record reproduce, σ = 0.1.
* **Phase 2b** (`ko_sel_s0`, `ko_ctl_s0`): stronger selection. 32 genomes, knockout bracket (one game per match), the top 8 also play a qualifier vs the random mover and a hall of fame of past champions, **only the best 4 reproduce**, σ = 0.03.
* **Phase 2c** (`c_sel_s0`, `c_ctl_s0`, current defaults): **pass only when no other move is possible**, 64 genomes, qualifier vs past champions only (the random mover is just the yardstick), 2000 generations, numba engine (~7× faster).
* **Controls** (`*ctl*`): identical, except the parents are picked at random.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"   # or "claude/trusting-keller-qlgif5" until merged
if IN_COLAB:
    if not os.path.exists("go-evo"):
        !git clone -q --depth 1 -b {BRANCH} https://github.com/PrimozRavbar/go-evo.git
    %cd go-evo
    !pip install -q -r requirements.txt
else:
    os.chdir(os.path.abspath(".."))  # repo root
    sys.path.insert(0, os.getcwd())
import numpy as np, matplotlib.pyplot as plt
from goevo import NetConfig, GameConfig, Player, play_game, load_population
from goevo.evolve import EvoConfig, run_evolution, load_history
from goevo.analysis import bench_curve, survivor_advantage, head_to_head, plot_runs
from goevo.viz import animate_game
print("CPUs:", os.cpu_count())

## A. Results of the pushed runs (`runs/`)
Only the final checkpoint of each run is kept (phase 2b: just the 4 parents). Generation 0 is regenerated from the seed.

In [ ]:
from goevo.analysis import plot_progress
from IPython.display import Image, display
C_RUNS = {k: f"runs/{k}" for k in ["c_sel_s0", "c_ctl_s0"] if os.path.exists(f"runs/{k}/history.jsonl")}
if C_RUNS:
    plot_progress(C_RUNS, "/tmp/progress.png"); display(Image("/tmp/progress.png"))
RUNS = {k: f"runs/{k}" for k in ["ko_sel_s0", "ko_ctl_s0", "sel_s0", "sel_s1", "ctl_s0", "ctl_s1"]
        if os.path.exists(f"runs/{k}/history.jsonl")}
plot_runs(RUNS, "/tmp/runs.png")
from IPython.display import Image, display
display(Image("/tmp/runs.png"))
for k, d in RUNS.items():
    h = load_history(d); g, m = bench_curve(h)
    print(f"{k}: bench win-rate gen0={m[0]:.3f}  last-5-avg={m[-5:].mean():.3f}  ({len(h)-1} generations)")

## B. Watch: evolved champion vs a generation-0 genome

In [ ]:
import json
from goevo.evolve import EvoConfig, initial_population
RUN = "runs/c_sel_s0" if os.path.exists("runs/c_sel_s0") else "runs/ko_sel_s0"
ckpt = sorted(f for f in os.listdir(RUN) if f.endswith(".pt"))[-1]
final, net_cfg, names, extra = load_population(f"{RUN}/{ckpt}")
cfg = json.load(open(f"{RUN}/config.json"))["evo"]
gen0 = [g.genome for g in initial_population(EvoConfig(pop_size=cfg["pop_size"], seed=cfg["seed"]), net_cfg)[0]]
game_cfg = GameConfig(board_size=net_cfg.board_size)   # fast engine, pass only when forced
champ = Player(final[0], net_cfg, name=f"final {names[0]}")   # strongest first
old = Player(gen0[0], net_cfg, name="gen0 #0")
print(RUN, ckpt, "| final genomes:", len(final), "| gen0 genomes:", len(gen0))
rec = play_game(champ, old, game_cfg)
print(rec.black, "vs", rec.white, "->", rec.result_string, len(rec.moves), "moves")
animate_game(rec, interval_ms=80, every=2)  # every=1 for every move

## C. Head-to-head: final genomes vs generation-0 population (all pairs, both colours)

In [ ]:
wr, n = head_to_head(final, gen0, net_cfg, game_cfg, workers=os.cpu_count())
print(f"final vs gen0: win rate {wr:.3f} over {n} games")

## D. Run your own (short) evolution here

In [ ]:
evo = EvoConfig(pop_size=64, generations=20, bench_every=5, checkpoint_every=0,
                out_dir="runs/colab_demo", workers=os.cpu_count())   # add resume=True to continue
hist = run_evolution(evo, NetConfig(board_size=19), GameConfig(board_size=19))